In [1]:
from __future__ import annotations

from datetime import datetime
from dateutil.relativedelta import relativedelta
from pathlib import Path
import re

import pandas as pd


In [3]:
from src.config.config import load_equipment_config, DATA_BASE_DIR
from src.domain.entities.equipment import Equipment, create_equipment
EQUIPMENTS = load_equipment_config(Path("src/config/equipments.yaml"))

In [4]:
from src.databases import OracleCRUD, PostgresCRUD
from ftplib import FTP, all_errors
# from src.databases.postgresql_db import PostgresCRUD

In [ ]:
devices = ["SW2L", "AG14"]
device_query = ",".join([f"'{device}'" for device in devices])
query = f"""
SELECT DISTINCT
    MAT_ID product, MAT_CMF_21 device, MAT_CMF_5 device_name,
    MAT_CMF_6 tech_prod, MAT_CMF_7 tech, MAT_CMF_23 prod_group
FROM IDBMGR.MWIPMATDEF
WHERE FACTORY = 'AFB1'
    AND MAT_CMF_21 IN ({device_query}) 
ORDER BY MAT_CMF_21
"""
df_tech = OracleCRUD.select_query(query)
if df_tech.empty:
    
df_tech["result"] = df_tech.apply(
    lambda row: (
        True
        if row["device_name"] in [None, "", " "]
        else row["device"] in row["product"]
    ),
    axis=1,
)
df_tech.columns = df_tech.columns.str.lower()
df_tech = df_tech[df_tech["result"]].drop(
    ["product", "device_name", "result"], axis=1
)
df_tech = df_tech.drop_duplicates()
df_tech

,device,tech_prod,tech,prod_group
0,AG14,C05,0.18,BCDMOS
6,SW2L,C05,0.16,LDI


In [4]:
equip = create_equipment(EQUIPMENTS[3])

In [6]:
def parse_edit_time(line: str) -> str:
    """파일 정보 문자 내 최종 수정 일자 파싱"""
    # 연도가 없고 날짜만 있는 경우 : 오늘 월/일보다 뒤에 있을 경우 작년, 오늘 월/일보다 앞에 있을 경우 올해
    pattern_1 = r"(\b[A-Za-z]{3}\s\d{2}\s\d{2}:\d{2}\b)"
    pattern_2 = r"(\b[A-Za-z]{3}\s\s\d{1}\s\d{2}:\d{2}\b)"
    pattern_3 = r"(\b[A-Za-z]{3}\s\d{2}\s\s\d{4}\b)"
    pattern_4 = r"(\b[A-Za-z]{3}\s\s\d{1}\s\s\d{4}\b)"
    pattern_5 = r"(\d+[가-힣]\s\d+[가-힣]\s\d{4}[가-힣])"
    pattern_6 = r"(\d+[가-힣]\s\s\d+[가-힣]\s\d{4}[가-힣])"
    pattern_7 = r"(\d+[가-힣]\s\d+[가-힣]\s\s\d{2}:\d{2}\b)"
    try:
        info = re.findall(pattern_1, line)[0]
        year = datetime.now().year
        edit_time = datetime.strptime(f"{year} {info}", "%Y %b %d %H:%M")
    except:
        try:
            info = re.findall(pattern_2, line)[0]
            year = datetime.now().year
            edit_time = datetime.strptime(f"{year} {info}", "%Y %b  %d %H:%M")
        except:
            try:
                info = re.findall(pattern_3, line)[0]
                edit_time = datetime.strptime(info, "%b %d  %Y")
            except:
                try:
                    info = re.findall(pattern_4, line)[0]
                    edit_time = datetime.strptime(info, "%b  %d  %Y")
                except:
                    try:
                        info = re.findall(pattern_5, line)[0]
                        info = "-".join(
                            [
                                re.match(r"\d+", item).group(0)
                                for item in info.split(" ")
                            ]
                        )
                        edit_time = datetime.strptime(info, "%m-%d-%Y")
                    except:
                        try:
                            info = re.findall(pattern_6, line)[0]
                            info = "-".join(
                                [
                                    re.match(r"\d+", item).group(0)
                                    for item in info.split(" ")
                                    if item is not None and len(item) > 0
                                ]
                            )
                            edit_time = datetime.strptime(info, "%m-%d-%Y")
                        except:
                            try:
                                info = re.findall(pattern_7, line)[0]
                                info = re.sub("[가-힣]", " ", info)
                                edit_time = datetime.strptime(
                                    f"{datetime.now().year} {info}", "%Y %m  %d   %H:%M"
                                )
                            except:
                                print(line)
                                edit_time = datetime.now()
    if edit_time > datetime.now():
        edit_time = edit_time - relativedelta(years=1)
    return edit_time.strftime("%Y-%m-%d %H:%M")


def recipe_filter(file: str) -> bool:
    """Recipe 폴더 필터

    Args:
        file (str): 파일명

    Returns:
        bool: 필터 통과 여부
    """
    # .이나 _로 시작하거나 특정 문자/확장자 로 끝나는 경우 제외
    if file.startswith((".", "_")) or file.endswith(("~", ".inf", ".DAT")):
        return False

    # 언더바(_)로 분할했을 때 정확히 2개여야 하고, 앞부분의 길이가 4여야 함
    parts = file.split("_")
    return len(parts) == 2 and len(parts[0]) == 4

In [18]:
def collect_lines(ftp: FTP) -> List[str]:
    """FTP 폴더 내 파일 정보 수집

    Args:
        ftp (FTP): 접속한 ftp 인스턴스

    Returns:
        List[str]: 폴더/파일 리스트
    """
    lines = []
    def _collect(line):
        lines.append(line)
    ftp.dir(_collect)
    return lines

In [ ]:
def is_recipe(line: str) -> bool:
    """Recipe 폴더 필터

    Args:
        line (str): 파일 정보

    Returns:
        bool: 필터 통과 여부
    """
    info = line.split(" ")
    file_name = info[-1].strip()
    # .이나 _로 시작하거나 특정 문자/확장자 로 끝나는 경우 제외
    if file_name.startswith((".", "_")) or file_name.endswith(("~", ".inf", ".DAT")):
        return False

    # 언더바(_)로 분할했을 때 정확히 2개여야 하고, 앞부분의 길이가 4여야 함
    parts = file_name.split("_")
    return len(parts) == 2 and len(parts[0]) == 4

def transform_recipe_info(
    line: str, equip: Equipment, equip_dir: Path
    ) -> Tuple[str, str, str, str, str, str]:
    """Recipe 정보 추출

    Args:
        line (str): 파일 정보

    Returns:
        Tuple[str, str, str, str, str, str]:
        Recipe Key, Device, Layer, Recipe, Edit Time
    """
    info = line.split(" ")
    file_name = info[-1].strip()
    device, layer = file_name.split("_")
    last_edit_time = parse_edit_time(line)
    recipe_key = f"{equip.name}-{file_name}"
    backup_recipe_dir = equip_dir / file_name
    backup_recipe_path = backup_recipe_dir / "ins.dat"
    return (recipe_key, device, layer, file_name, last_edit_time,backup_recipe_dir, backup_recipe_path)


def is_file_update(equip_db: pd.DataFrame, rcp_key: str, edit_time: datetime, backup_file_path: Path):
    """DB 정보와 파일 최종 수정시간 비교하여 파일 업데이트 상태 최신화"""
    match_info = equip_db[equip_db["recipe_key"] == rcp_key]
    if match_info.empty:
        return True, "new"  # New (신규 Recipe)
    if not backup_file_path.exists():
        return True, "new"
    db_edit_time = match_info.iloc[0]["dt_edit_time"]

    if edit_time > db_edit_time:
        return True, "update"  # Update (수정된 Recipe)
    return False, None
    

In [41]:
equip_dir = DATA_BASE_DIR / equip.name

In [38]:
df_equip_db = PostgesCRUD.select_query(
    """
	SELECT recipe_key, dt_edit_time, device, layer, recipe, backup_file_path
	FROM dm_recipe_info
	WHERE equip = %s
		AND is_remove=false
	""",
 (equip.name,))

In [ ]:
with FTP(
        timeout=equip.ftp_info.timeout_seconds, encoding=equip.ftp_info.encoding
    ) as ftp:
    ftp.connect(host=equip.ftp_info.host, port=equip.ftp_info.port)
    ftp.login(equip.ftp_info.username, equip.ftp_info.password)
    ftp.cwd(equip.ftp_info.remote_directory)
    lines = collect_lines(ftp)
    recipes = [transform_recipe_info(line, equip) for line in lines if is_recipe(line)]
    for i, (rcp_key, device, layer, recipe, edit_time,
            backup_recipe_dir, backup_recipe_path) in enumerate(recipes):
        is_upsert, file_status = is_file_update(df_equip_db, rcp_key, edit_time, backup_recipe_path)
        if is_upsert:
            backup_recipe_dir.mkdir(parents=True, exists=True)
            ftp_download_tree(ftp, recipe, backup_recipe_dir)
            print(rcp_key, equip.model, equip.name, device, layer, recipe, edit_time)
        if i>5:
            break

SMDI06-A002_1565PAD3 KLA2139 SMDI06 A002 1565PAD3 A002_1565PAD3 2026-09-05 16:10
SMDI06-A003_118062UM KLA2139 SMDI06 A003 118062UM A003_118062UM 2026-09-05 16:10
SMDI06-A003_1400MOPT KLA2139 SMDI06 A003 1400MOPT A003_1400MOPT 2026-09-05 16:10
SMDI06-A003_145062UM KLA2139 SMDI06 A003 145062UM A003_145062UM 2026-09-05 16:10
SMDI06-A003_1565PAD3 KLA2139 SMDI06 A003 1565PAD3 A003_1565PAD3 2026-09-05 16:10
SMDI06-A003_3365GATE KLA2139 SMDI06 A003 3365GATE A003_3365GATE 2026-09-05 16:10
SMDI06-A003_5085COSI KLA2139 SMDI06 A003 5085COSI A003_5085COSI 2026-09-05 16:10


In [ ]:
"/" + (DATA_BASE_DIR / equip.name / "ins.dat").relative_to(DATA_BASE_DIR)

TypeError: can only concatenate str (not "WindowsPath") to str

In [37]:
for (rcp_key, device, layer, recipe, edit_time) in recipes:
    print(rcp_key, equip.model, equip.name, device, layer, recipe, edit_time)
    

SMDI06-A002_1565PAD3 KLA2139 SMDI06 A002 1565PAD3 A002_1565PAD3 2026-09-05 16:10
SMDI06-A003_118062UM KLA2139 SMDI06 A003 118062UM A003_118062UM 2026-09-05 16:10
SMDI06-A003_1400MOPT KLA2139 SMDI06 A003 1400MOPT A003_1400MOPT 2026-09-05 16:10
SMDI06-A003_145062UM KLA2139 SMDI06 A003 145062UM A003_145062UM 2026-09-05 16:10
SMDI06-A003_1565PAD3 KLA2139 SMDI06 A003 1565PAD3 A003_1565PAD3 2026-09-05 16:10
SMDI06-A003_3365GATE KLA2139 SMDI06 A003 3365GATE A003_3365GATE 2026-09-05 16:10
SMDI06-A003_5085COSI KLA2139 SMDI06 A003 5085COSI A003_5085COSI 2026-09-05 16:10
SMDI06-A003_537562UM KLA2139 SMDI06 A003 537562UM A003_537562UM 2026-09-05 16:10
SMDI06-A003_615562UM KLA2139 SMDI06 A003 615562UM A003_615562UM 2026-09-05 16:10
SMDI06-A003_616562UM KLA2139 SMDI06 A003 616562UM A003_616562UM 2026-09-05 16:10
SMDI06-A003_6600M2PT KLA2139 SMDI06 A003 6600M2PT A003_6600M2PT 2026-09-05 16:10
SMDI06-A003_665562UM KLA2139 SMDI06 A003 665562UM A003_665562UM 2026-09-05 16:10
SMDI06-A003_6900V2PT KLA2139

In [39]:
df_equip_db

,recipe_key,dt_edit_time,device,layer,recipe,backup_file_path
